In [1]:
# Load and save global mortality in single files

In [2]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [3]:
# Number of samples: full distribution or stats
n_samples = 300

In [5]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop - 1}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/global/{n_samples}_samples/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/"

for ens_num in ensemble_members:
    print(f"Processing ensemble number {ens_num:02d}")

    # === FULL DIST. OR JUST STATS === 
    if n_samples <= 500:
        files = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*.nc"
        description = ("Global mortality (COPD) due to ozone "
                       " - scripts by A.F. Wells (2025)")
        out_file = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

    else:
        files = f"Global_mortality_stats_{model}_{scenario}_{ens_num:02d}_*.nc"
        description = ("Global mortality (COPD) due to ozone "
                       "statistics: including mean, median, "
                       "and the 95% CI - scripts by A.F. Wells (2025)")
        out_file = f"Global_mortality_stats_{model}_{scenario}_{ens_num:02d}_{dates}.nc"

    file_path = os.path.join(MORT_DIR, files)

    ds = xr.open_mfdataset(
        sorted(glob.glob(file_path)),
        combine="nested",
        concat_dim="year")
    ds = ds.assign_coords(year=years)  # Last year removed from OSDMA8

    ds.attrs["description"] = description
    ds.attrs["model"] = model
    ds.attrs["scenario"] = scenario
    ds.attrs["ensemble_number"] = ens_num

    print(f"Saving {out_file}")
    out_path = os.path.join(SAVE_DIR, out_file)
    ds.to_netcdf(out_path)

print("All processing complete.")

Processing ensemble number 01
Saving Global_mortality_300samples_UKESM1_SSP245_G6_01_2020-2083.nc
Processing ensemble number 02
Saving Global_mortality_300samples_UKESM1_SSP245_G6_02_2020-2083.nc
Processing ensemble number 03
Saving Global_mortality_300samples_UKESM1_SSP245_G6_03_2020-2083.nc
All processing complete.
